In [35]:
import pandas as pd

df = pd.read_csv('../data/raw/DataCoSupplyChainDataset.csv', encoding='unicode_escape')

print("Shape:", df.shape)
print("\nColumns:")
for col in df.columns:
    print(" ", col)

Shape: (180519, 53)

Columns:
  Type
  Days for shipping (real)
  Days for shipment (scheduled)
  Benefit per order
  Sales per customer
  Delivery Status
  Late_delivery_risk
  Category Id
  Category Name
  Customer City
  Customer Country
  Customer Email
  Customer Fname
  Customer Id
  Customer Lname
  Customer Password
  Customer Segment
  Customer State
  Customer Street
  Customer Zipcode
  Department Id
  Department Name
  Latitude
  Longitude
  Market
  Order City
  Order Country
  Order Customer Id
  order date (DateOrders)
  Order Id
  Order Item Cardprod Id
  Order Item Discount
  Order Item Discount Rate
  Order Item Id
  Order Item Product Price
  Order Item Profit Ratio
  Order Item Quantity
  Sales
  Order Item Total
  Order Profit Per Order
  Order Region
  Order State
  Order Status
  Order Zipcode
  Product Card Id
  Product Category Id
  Product Description
  Product Image
  Product Name
  Product Price
  Product Status
  shipping date (DateOrders)
  Shipping Mode


In [36]:
with open('../data/columns_and_nulls.txt', 'w') as f:
    f.write("SHAPE\n")
    f.write(f"{df.shape}\n\n")
    
    f.write("ALL COLUMNS\n")
    for i, col in enumerate(df.columns):
        f.write(f"{i:02d}. {col}\n")
    
    f.write("\nNULL COUNTS\n")
    nulls = df.isnull().sum()[df.isnull().sum() > 0]
    f.write(str(nulls))
    
    f.write("\n\nDATA TYPES\n")
    f.write(str(df.dtypes))

print("Saved to data/columns_and_nulls.txt")

Saved to data/columns_and_nulls.txt


In [37]:
import sqlite3

conn = sqlite3.connect('../data/supply_chain.db')
cursor = conn.cursor()

with open('../sql/01_schema.sql', 'r') as f:
    sql = f.read()

# Execute only non-comment statements
conn.executescript(sql)
conn.commit()

print("Schema created successfully.")

# Verify all tables exist
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tables = cursor.fetchall()
print("\nTables created:")
for t in tables:
    print(" ", t[0])

conn.close()

Schema created successfully.

Tables created:
  dim_customer
  dim_date
  dim_location
  dim_product
  fact_orders
  pipeline_run_log
  sqlite_sequence


In [38]:
import pandas as pd
import sqlite3
import logging
from datetime import datetime

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s — %(levelname)s — %(message)s'
)
logger = logging.getLogger(__name__)

print("Imports loaded.")

Imports loaded.


In [39]:
# Define paths
CSV_PATH = '../data/raw/DataCoSupplyChainDataset.csv'
DB_PATH  = '../data/supply_chain.db'

print(f"CSV: {CSV_PATH}")
print(f"DB:  {DB_PATH}")

CSV: ../data/raw/DataCoSupplyChainDataset.csv
DB:  ../data/supply_chain.db


In [40]:
# ETL pipeline : EXTRACT
def extract(csv_path):
    logger.info("EXTRACT — loading raw CSV...")
    df = pd.read_csv(csv_path, encoding='unicode_escape')
    logger.info(f"EXTRACT — {len(df):,} rows, {len(df.columns)} columns loaded")
    return df

df_raw = extract(CSV_PATH)
df_raw.head(2)

2026-05-25 20:06:27,614 — INFO — EXTRACT — loading raw CSV...
2026-05-25 20:06:28,384 — INFO — EXTRACT — 180,519 rows, 53 columns loaded


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class


In [41]:
#ETL pipeline: TRANSFORM
def transform(df):
    logger.info("TRANSFORM — cleaning data...")
    df = df.copy()

    df.columns = df.columns.str.strip()

    drop_cols = [
        'Customer Email', 'Customer Password',
        'Product Description', 'Product Image', 'Order Zipcode'
    ]
    df.drop(columns=drop_cols, inplace=True, errors='ignore')

    df['days_late'] = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']
    df['is_late']   = (df['days_late'] > 0).astype(int)

    df['Customer Lname']    = df['Customer Lname'].fillna('Unknown')
    df['Customer Zipcode']  = df['Customer Zipcode'].fillna(0).astype(int).astype(str)

    logger.info(f"TRANSFORM — done. Shape: {df.shape}")
    return df

df_clean = transform(df_raw)

print("Derived columns added:")
print(df_clean[['Days for shipping (real)', 'Days for shipment (scheduled)', 'days_late', 'is_late']].head(5))

2026-05-25 20:06:31,706 — INFO — TRANSFORM — cleaning data...
2026-05-25 20:06:31,848 — INFO — TRANSFORM — done. Shape: (180519, 50)


Derived columns added:
   Days for shipping (real)  Days for shipment (scheduled)  days_late  is_late
0                         3                              4         -1        0
1                         5                              4          1        1
2                         4                              4          0        0
3                         3                              4         -1        0
4                         2                              4         -2        0


In [44]:
#ETL pipeline: LOAD

conn = sqlite3.connect(DB_PATH)
print("Connection reopened.")

def load_dim_date(df, conn):
    logger.info("LOAD — dim_date...")

    # Parse and extract DATE only — drop time component
    dates = pd.to_datetime(
        df['order date (DateOrders)'], errors='coerce'
    ).dt.normalize().dropna().unique()

    date_df = pd.DataFrame({'full_date': sorted(dates)})
    date_df['full_date']   = pd.to_datetime(date_df['full_date'])
    date_df['date_id']     = date_df.index + 1
    date_df['year']        = date_df['full_date'].dt.year
    date_df['month']       = date_df['full_date'].dt.month
    date_df['month_name']  = date_df['full_date'].dt.strftime('%B')
    date_df['quarter']     = date_df['full_date'].dt.quarter
    date_df['day']         = date_df['full_date'].dt.day
    date_df['day_of_week'] = date_df['full_date'].dt.strftime('%A')
    date_df['is_weekend']  = date_df['full_date'].dt.dayofweek.isin([5,6]).astype(int)
    date_df['full_date']   = date_df['full_date'].dt.strftime('%Y-%m-%d')

    date_df.to_sql('dim_date', conn, if_exists='replace', index=False)
    logger.info(f"LOAD — dim_date: {len(date_df):,} rows inserted")
    return date_df

date_df = load_dim_date(df_clean, conn)
print(f"dim_date rows: {len(date_df):,}")
date_df.head(3)

2026-05-25 20:09:04,877 — INFO — LOAD — dim_date...


Connection reopened.


2026-05-25 20:09:05,191 — INFO — LOAD — dim_date: 1,127 rows inserted


dim_date rows: 1,127


,full_date,date_id,year,month,month_name,quarter,day,day_of_week,is_weekend
0,2015-01-01,1,2015,1,January,1,1,Thursday,0
1,2015-01-02,2,2015,1,January,1,2,Friday,0
2,2015-01-03,3,2015,1,January,1,3,Saturday,1


In [45]:
# Loading dim_customer
def load_dim_customer(df, conn):
    logger.info("LOAD — dim_customer...")
    customer_df = df[[
        'Customer Id', 'Customer Fname', 'Customer Lname',
        'Customer Segment', 'Customer City', 'Customer State',
        'Customer Country', 'Customer Street', 'Customer Zipcode', 'Market'
    ]].drop_duplicates(subset='Customer Id').copy()

    customer_df.columns = [
        'customer_id', 'first_name', 'last_name',
        'segment', 'city', 'state',
        'country', 'street', 'zipcode', 'market'
    ]

    customer_df.to_sql('dim_customer', conn, if_exists='replace', index=False)
    logger.info(f"LOAD — dim_customer: {len(customer_df):,} rows inserted")
    return customer_df

customer_df = load_dim_customer(df_clean, conn)
customer_df.head(3)

2026-05-25 20:09:10,601 — INFO — LOAD — dim_customer...
2026-05-25 20:09:10,661 — INFO — LOAD — dim_customer: 20,652 rows inserted


,customer_id,first_name,last_name,segment,city,state,country,street,zipcode,market
0,20755,Cally,Holloway,Consumer,Caguas,PR,Puerto Rico,5365 Noble Nectar Island,725,Pacific Asia
1,19492,Irene,Luna,Consumer,Caguas,PR,Puerto Rico,2679 Rustic Loop,725,Pacific Asia
2,19491,Gillian,Maldonado,Consumer,San Jose,CA,EE. UU.,8510 Round Bear Gate,95125,Pacific Asia


In [46]:
# Loading dim_product
def load_dim_customer(df, conn):
    logger.info("LOAD — dim_customer...")
    customer_df = df[[
        'Customer Id', 'Customer Fname', 'Customer Lname',
        'Customer Segment', 'Customer City', 'Customer State',
        'Customer Country', 'Customer Street', 'Customer Zipcode', 'Market'
    ]].drop_duplicates(subset='Customer Id').copy()

    customer_df.columns = [
        'customer_id', 'first_name', 'last_name',
        'segment', 'city', 'state',
        'country', 'street', 'zipcode', 'market'
    ]

    customer_df.to_sql('dim_customer', conn, if_exists='replace', index=False)
    logger.info(f"LOAD — dim_customer: {len(customer_df):,} rows inserted")
    return customer_df

customer_df = load_dim_customer(df_clean, conn)
customer_df.head(3)

2026-05-25 20:09:15,335 — INFO — LOAD — dim_customer...
2026-05-25 20:09:15,403 — INFO — LOAD — dim_customer: 20,652 rows inserted


,customer_id,first_name,last_name,segment,city,state,country,street,zipcode,market
0,20755,Cally,Holloway,Consumer,Caguas,PR,Puerto Rico,5365 Noble Nectar Island,725,Pacific Asia
1,19492,Irene,Luna,Consumer,Caguas,PR,Puerto Rico,2679 Rustic Loop,725,Pacific Asia
2,19491,Gillian,Maldonado,Consumer,San Jose,CA,EE. UU.,8510 Round Bear Gate,95125,Pacific Asia


In [47]:
# Loading dim_product
def load_dim_product(df, conn):
    logger.info("LOAD — dim_product...")
    product_df = df[[
        'Product Card Id', 'Product Name', 'Product Category Id',
        'Category Name', 'Department Id', 'Department Name',
        'Product Price', 'Product Status'
    ]].drop_duplicates(subset='Product Card Id').copy()

    product_df.columns = [
        'product_id', 'product_name', 'category_id',
        'category_name', 'department_id', 'department_name',
        'product_price', 'product_status'
    ]

    product_df.to_sql('dim_product', conn, if_exists='replace', index=False)
    logger.info(f"LOAD — dim_product: {len(product_df):,} rows inserted")
    return product_df

product_df = load_dim_product(df_clean, conn)
product_df.head(3)

2026-05-25 20:09:18,869 — INFO — LOAD — dim_product...
2026-05-25 20:09:18,889 — INFO — LOAD — dim_product: 118 rows inserted


,product_id,product_name,category_id,category_name,department_id,department_name,product_price,product_status
0,1360,Smart watch,73,Sporting Goods,2,Fitness,327.750000,0
48,365,Perfect Fitness Perfect Rip Deck,17,Cleats,4,Apparel,59.990002,0
49,627,Under Armour Girls' Toddler Spine Surge Runni,29,Shop By Sport,5,Golf,39.990002,0


In [49]:
def load_dim_location(df, conn):
    logger.info("LOAD — dim_location...")

    # Group by meaningful columns, average the lat/long variations
    location_df = df.groupby(
        ['Order City', 'Order State', 'Order Country', 'Order Region', 'Shipping Mode'],
        as_index=False
    ).agg(
        latitude=('Latitude', 'mean'),
        longitude=('Longitude', 'mean')
    )

    location_df = location_df.reset_index(drop=True)
    location_df['location_id'] = location_df.index + 1

    location_df.columns = [
        'order_city', 'order_state', 'order_country',
        'order_region', 'shipping_mode',
        'latitude', 'longitude', 'location_id'
    ]

    location_df.to_sql('dim_location', conn, if_exists='replace', index=False)
    logger.info(f"LOAD — dim_location: {len(location_df):,} rows inserted")
    return location_df

conn        = sqlite3.connect(DB_PATH)
location_df = load_dim_location(df_clean, conn)
print(f"dim_location rows: {len(location_df):,}")
location_df.head(3)

2026-05-25 20:09:29,956 — INFO — LOAD — dim_location...
2026-05-25 20:09:30,029 — INFO — LOAD — dim_location: 9,530 rows inserted


dim_location rows: 9,530


,order_city,order_state,order_country,order_region,shipping_mode,latitude,longitude,location_id
0,Aachen,Renania del Norte-Westfalia,Alemania,Western Europe,First Class,33.439702,-107.032265,1
1,Aachen,Renania del Norte-Westfalia,Alemania,Western Europe,Same Day,28.577641,-81.447083,2
2,Aachen,Renania del Norte-Westfalia,Alemania,Western Europe,Second Class,32.749722,-117.002258,3


In [50]:
# Loading fact_orders
def load_fact_orders(df, conn, date_df, location_df):
    logger.info("LOAD — fact_orders...")

    date_map = dict(zip(date_df['full_date'], date_df['date_id']))
    dates = pd.to_datetime(df['order date (DateOrders)'], errors='coerce').dt.strftime('%Y-%m-%d')
    df = df.copy()
    df['date_id'] = dates.map(date_map)

    df = df.merge(
        location_df[['order_city','order_state','order_country',
                     'order_region','shipping_mode','location_id']],
        left_on=['Order City','Order State','Order Country',
                 'Order Region','Shipping Mode'],
        right_on=['order_city','order_state','order_country',
                  'order_region','shipping_mode'],
        how='left'
    )

    fact_df = df[[
        'Order Item Id', 'Order Id', 'date_id', 'Customer Id',
        'Product Card Id', 'location_id', 'Sales',
        'Order Item Total', 'Benefit per order',
        'Order Profit Per Order', 'Sales per customer',
        'Order Item Discount', 'Order Item Discount Rate',
        'Order Item Profit Ratio', 'Order Item Quantity',
        'Order Item Product Price', 'Days for shipping (real)',
        'Days for shipment (scheduled)', 'days_late', 'is_late',
        'Late_delivery_risk', 'Delivery Status', 'Order Status',
        'shipping date (DateOrders)', 'order date (DateOrders)', 'Type'
    ]].copy()

    fact_df.columns = [
        'order_item_id', 'order_id', 'date_id', 'customer_id',
        'product_id', 'location_id', 'sales_amount',
        'order_item_total', 'benefit_per_order',
        'order_profit_per_order', 'sales_per_customer',
        'order_item_discount', 'order_item_discount_rate',
        'order_item_profit_ratio', 'order_item_quantity',
        'order_item_product_price', 'days_shipping_real',
        'days_shipping_scheduled', 'days_late', 'is_late',
        'late_delivery_risk', 'delivery_status', 'order_status',
        'shipping_date', 'order_date', 'order_type'
    ]

    fact_df.to_sql('fact_orders', conn, if_exists='replace', index=False)
    logger.info(f"LOAD — fact_orders: {len(fact_df):,} rows inserted")
    return fact_df

fact_df = load_fact_orders(df_clean, conn, date_df, location_df)
fact_df.head(3)

2026-05-25 20:09:31,632 — INFO — LOAD — fact_orders...
2026-05-25 20:09:32,375 — INFO — LOAD — fact_orders: 180,519 rows inserted


,order_item_id,order_id,date_id,customer_id,product_id,location_id,sales_amount,order_item_total,benefit_per_order,order_profit_per_order,...,days_shipping_real,days_shipping_scheduled,days_late,is_late,late_delivery_risk,delivery_status,order_status,shipping_date,order_date,order_type
0,180517,77202,1127,20755,1360,878,327.75,314.640015,91.250000,91.250000,...,3,4,-1,0,0,Advance shipping,COMPLETE,2/3/2018 22:56,1/31/2018 22:56,DEBIT
1,179254,75939,1109,19492,1360,1029,327.75,311.359985,-249.089996,-249.089996,...,5,4,1,1,1,Late delivery,PENDING,1/18/2018 12:27,1/13/2018 12:27,TRANSFER
2,179253,75938,1109,19491,1360,1029,327.75,309.720001,-247.779999,-247.779999,...,4,4,0,0,0,Shipping on time,CLOSED,1/17/2018 12:06,1/13/2018 12:06,CASH


In [51]:
def log_pipeline_run(conn, status, rows_loaded, error=None):
    log_df = pd.DataFrame([{
        'run_timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'status':        status,
        'rows_loaded':   rows_loaded,
        'error_message': error or ''
    }])
    log_df.to_sql('pipeline_run_log', conn, if_exists='append', index=False)
    logger.info(f"PIPELINE LOG — status: {status}, rows: {rows_loaded:,}")

log_pipeline_run(conn, 'SUCCESS', len(fact_df))
conn.close()
logger.info("DATABASE — connection closed")

2026-05-25 20:09:38,503 — INFO — PIPELINE LOG — status: SUCCESS, rows: 180,519
2026-05-25 20:09:38,504 — INFO — DATABASE — connection closed


In [52]:
conn = sqlite3.connect(DB_PATH)

tables = ['dim_date', 'dim_customer', 'dim_product', 'dim_location', 'fact_orders', 'pipeline_run_log']

print("Table row counts:")
print("-" * 35)
for table in tables:
    count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table}", conn).iloc[0]['cnt']
    print(f"  {table:<25} {count:>10,}")

conn.close()

Table row counts:
-----------------------------------
  dim_date                       1,127
  dim_customer                  20,652
  dim_product                      118
  dim_location                   9,530
  fact_orders                  180,519
  pipeline_run_log                   3


In [55]:
conn = sqlite3.connect(DB_PATH)

checks = {
    "Orphaned date_id":     "SELECT COUNT(*) FROM fact_orders WHERE date_id NOT IN (SELECT date_id FROM dim_date)",
    "Orphaned customer_id": "SELECT COUNT(*) FROM fact_orders WHERE customer_id NOT IN (SELECT customer_id FROM dim_customer)",
    "Orphaned product_id":  "SELECT COUNT(*) FROM fact_orders WHERE product_id NOT IN (SELECT product_id FROM dim_product)",
    "Orphaned location_id": "SELECT COUNT(*) FROM fact_orders WHERE location_id NOT IN (SELECT location_id FROM dim_location)",
}

print("Referential integrity checks:")
print("-" * 40)
all_passed = True
for check_name, query in checks.items():
    result = pd.read_sql(query, conn).iloc[0, 0]
    status = "PASS" if result == 0 else f"FAIL — {result:,} orphaned rows"
    if result != 0:
        all_passed = False
    print(f"  {check_name:<25} {status}")

print("-" * 40)
print(f"  Overall: {'ALL CHECKS PASSED' if all_passed else 'ISSUES FOUND'}")

conn.close()

Referential integrity checks:
----------------------------------------
  Orphaned date_id          PASS
  Orphaned customer_id      PASS
  Orphaned product_id       PASS
  Orphaned location_id      PASS
----------------------------------------
  Overall: ALL CHECKS PASSED
